# 10 - Instance Lifecycle Management

`IviumsoftInstanceManager` launches, tracks, adopts and closes IviumSoft processes,
mapping each OS process to the **driver instance number** it registers with. It is the
counterpart to the instance *scoping* in notebook `02`: scoping decides *which* running
instance a command targets; the manager decides *which instances exist*.

### When you need it

- Spin up N IviumSoft windows programmatically and drive them in parallel
- Recover after a crash: re-attach (`adopt`) to instances that outlived your script
- Clean up stray IviumSoft processes left behind by a previous run

### Notes

- **Windows only** - it uses native Win32 process helpers.
- The driver must be open. On a cold start with **no IviumSoft running yet**, open with
  `Pyvium.open_driver(verify_iviumsoft=False)` so the manager can launch the first one.
- Driver instance numbering is stable across a close: closing one leaves a gap rather
  than renumbering the survivors.

> This notebook drives real processes, so its cells are not pre-executed. Run it on a
> Windows machine with IviumSoft installed.

In [ ]:
from pyvium import Pyvium, IviumsoftInstanceManager
print("instance manager imported")

## 1. Cold start

Open the driver without requiring a running IviumSoft, then create the manager. If your
IviumSoft is not at the default `C:\\IviumStat\\IviumSoft.exe`, pass `exe_path=...`.

In [ ]:
Pyvium.open_driver(verify_iviumsoft=False)

manager = IviumsoftInstanceManager(
    # exe_path=r"C:\IviumStat\IviumSoft.exe",   # override if installed elsewhere
    launch_timeout=30.0,
    close_timeout=10.0,
)
print("driver open (cold start); manager ready")
print("already-active instances:", Pyvium.get_active_iviumsoft_instances())

## 2. Launch instances

`launch()` starts one IviumSoft process and blocks until it registers with the driver,
then returns its `ManagedInstance` (instance number + pid + launch time). It serializes
launches internally, so the instance number is attributed correctly even if you launch
several.

In [ ]:
first = manager.launch()
print("launched:", first)

second = manager.launch()
print("launched:", second)

print("instance numbers:", first.instance_number, second.instance_number)

## 3. List instances

`list_instances()` returns one record per active driver instance. Managed/adopted ones
carry a pid; instances this manager did not open (orphans) come back with `pid=None`.
Stale records (process gone) are pruned.

In [ ]:
for record in manager.list_instances():
    kind = "managed" if record.managed else ("adopted/orphan")
    print(f"  instance {record.instance_number}: pid={record.pid} ({kind})")

## 4. Drive a managed instance

The manager only handles lifecycle; use the scoping API from notebook `02` to send
commands. Here we connect the device on the first launched instance and read its status.

In [ ]:
handle = Pyvium.instance(first.instance_number)

status, label = handle.get_device_status()
print(f"instance {first.instance_number}: status ({status}, '{label}')")

if status == 0:  # IviumSoft up, device not yet connected
    try:
        handle.connect_device()
        print("  connected, serial:", handle.get_device_serial_number())
    except Exception as error:
        print(f"  connect skipped: {type(error).__name__}: {error}")

## 5. Discover: reconcile driver view vs OS view

`discover()` is read-only. It pairs what the manager tracks against what is actually
running, splitting out the two halves it cannot pair automatically: `orphan_instance_numbers`
(driver side) and `untracked_processes` (OS side). In a healthy state the counts match.

In [ ]:
report = manager.discover()
print("tracked:")
for record in report.tracked:
    print(f"  instance {record.instance_number} (pid {record.pid})")
print("orphan instance numbers:", report.orphan_instance_numbers)
print("untracked processes:")
for process in report.untracked_processes:
    print(f"  pid {process.pid}  started {process.started_at}  title {process.window_title!r}")

## 6. Adopt an instance the manager did not launch

After a script restart the IviumSoft windows are still running but this manager has no
record of them. `adopt(instance_number, pid)` re-attaches. The pid must come from an
external source you recorded, because the driver cannot map instance numbers to pids;
`discover()` helps you pair them by launch order (the driver numbers instances
sequentially).

In [ ]:
# Illustrative recovery flow: pair each untracked process (oldest first) with the
# lowest orphan instance number, then adopt it. Uncomment to run against real orphans.
#
# report = manager.discover()
# for instance_number, process in zip(report.orphan_instance_numbers,
#                                     report.untracked_processes):
#     record = manager.adopt(instance_number, process.pid)
#     print("adopted:", record)
print("adopt() re-attaches an existing instance by (instance_number, pid)")

## 7. Close an instance

`close()` sends a graceful window-close and escalates to a hard terminate if the process
does not exit within `close_timeout`. It refuses to close a *measuring* instance unless
`force=True`. Only instances with a known pid (launched or adopted) can be closed.

In [ ]:
manager.close(second.instance_number)
print("closed instance", second.instance_number)
print("remaining:", [r.instance_number for r in manager.list_instances()])

## 8. Sweep orphan processes

`close_orphans()` gracefully closes every IviumSoft process the manager does not track
(the `untracked_processes` of `discover()`). If any orphan instance is measuring, nothing
is closed unless `force=True` - prefer `discover()` + `adopt()` for instances still in use.

In [ ]:
# closed_pids = manager.close_orphans()          # add force=True to override a busy one
# print("closed orphan pids:", closed_pids)
print("close_orphans() sweeps untracked IviumSoft processes")

## Cleanup

Close whatever this notebook launched, then close the driver.

In [ ]:
for record in manager.list_instances():
    if record.managed:
        try:
            manager.close(record.instance_number, force=True)
            print("closed", record.instance_number)
        except Exception as error:
            print(f"close {record.instance_number} failed: {type(error).__name__}: {error}")

Pyvium.close_driver()
print("driver closed")

---

## Summary

| Task | API |
|------|-----|
| Cold-start open (no IviumSoft yet) | `Pyvium.open_driver(verify_iviumsoft=False)` |
| Create the manager | `IviumsoftInstanceManager(exe_path=..., ...)` |
| Launch an instance | `.launch()` -> `ManagedInstance` |
| List active instances | `.list_instances()` |
| Reconcile driver vs OS | `.discover()` -> `DiscoveryReport` |
| Re-attach after restart | `.adopt(instance_number, pid)` |
| Close one instance | `.close(instance_number, force=False)` |
| Sweep untracked processes | `.close_orphans(force=False)` |

## Next

- **`02_device_and_instance_management.ipynb`** - scope commands to an instance / channel
- **`08_batch_and_synchronization.ipynb`** - coordinate measurements across instances